#### Step 1 — Install libraries

#### Step 2 — Download helper scripts

In [1]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget $PREFIX/01-agentic-rag/code/rag_helper.py
!wget $PREFIX/04-evaluation/code/evaluation_utils.py

# Also download the embedder scripts from homework 2
EMBED = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget $EMBED/download.py
!wget $EMBED/embedder.py
!python download.py

--2026-07-17 14:41:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-17 14:41:16 (23.6 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-17 14:41:16--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response.

#### Step 3 — Set API key

In [3]:
import os

In [10]:
from openai import OpenAI

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("GROQ_API_KEY") or os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

In [5]:
#### Step 4 — Load the 72 lesson pages
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(len(documents))  # should be 72

72


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [12]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

##### Q1 — Generating Questions (Average Input Tokens)

In [13]:
from evaluation_utils import llm_structured, calc_total_price

# Only the first 3 pages
first_3 = [
    d for d in documents
    if d["filename"] in [
        "01-agentic-rag/lessons/01-intro.md",
        "01-agentic-rag/lessons/02-environment.md",
        "01-agentic-rag/lessons/03-rag.md",
    ]
]

input_tokens_list = []

for doc in first_3:
    user_prompt = f"filename: {doc['filename']}\n\ncontent: {doc['content']}"
    questions, usage = llm_structured(
         openai_client,
         data_gen_instructions,
         user_prompt,
         Questions,
         model= "openai/gpt-oss-20b"
    )
    input_tokens_list.append(usage.input_tokens)
    print(f"{doc['filename']}: {usage.input_tokens} input tokens")


avg = sum(input_tokens_list) / len(input_tokens_list)
print(f"\nAverage input tokens: {avg}")  # Q1 answer
 

01-agentic-rag/lessons/01-intro.md: 1089 input tokens
01-agentic-rag/lessons/02-environment.md: 1303 input tokens
01-agentic-rag/lessons/03-rag.md: 1715 input tokens

Average input tokens: 1369.0


In [23]:
input_tokens_list

[ResponseUsage(input_tokens=1089, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=562, output_tokens_details=OutputTokensDetails(reasoning_tokens=472), total_tokens=1651),
 ResponseUsage(input_tokens=1303, input_tokens_details=InputTokensDetails(cached_tokens=256), output_tokens=327, output_tokens_details=OutputTokensDetails(reasoning_tokens=200), total_tokens=1630),
 ResponseUsage(input_tokens=1715, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=302, output_tokens_details=OutputTokensDetails(reasoning_tokens=205), total_tokens=2017)]

In [14]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

import pandas as pd
df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")

print(len(ground_truth))           # should be 360
print(ground_truth[0])             # peek at first record

--2026-07-17 14:45:22--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.002s  

2026-07-17 14:45:22 (24.3 MB/s) - ‘ground-truth.csv’ saved [48627/48627]

360
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


#### Searching the chunks

##### Step 1 — Create chunks 

In [15]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))  # should be 295

295


##### Step 2 — Build text search index


In [16]:
from minsearch import  Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

##### Step 3 — Build vector search index

In [17]:
from embedder import Embedder
import numpy as np

embedder = Embedder()

# Embed all chunks (this takes a minute)
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

from minsearch import VectorSearch
vector_index = VectorSearch()
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)

2026-07-17 14:45:50.495837700 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


##### Step 4 — Build hybrid search

In [19]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    t = text_search(query, num_results=10)
    v = vector_search(query, num_results=10)
    return rrf([t, v], k=k)

#### Q2 — First Result with Text Search

In [20]:
q = ground_truth[0]["question"]
print("Question:", q)
print("Expected filename:", ground_truth[0]["filename"])

results = text_search(q)
print("Text search top result:", results[0]["filename"])  # Q2 answer

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Expected filename: 01-agentic-rag/lessons/01-intro.md
Text search top result: 01-agentic-rag/lessons/03-rag.md


#### Q3 — First Result with Vector Search

In [21]:
results = vector_search(q)
print("Vector search top result:", results[0]["filename"])  # Q3 answer

Vector search top result: 01-agentic-rag/lessons/01-intro.md


#### Evaluation metrics

In [22]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [24]:
from tqdm import tqdm
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [25]:
relevance_scores = compute_relevance_total(ground_truth, text_search)

100%|██████████| 360/360 [00:00<00:00, 517.58it/s]


##### Hit Rate

In [26]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [60]:
hit_rate_text = hit_rate(relevance_scores)
hit_rate_text

0.7583333333333333

##### Mean Reciprocal Rank (MRR)

In [27]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [62]:
mrr(relevance_scores)

0.5942592592592594

##### Putting it together

In [28]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

#### Q4. Evaluating text search

In [29]:
text_search_result = evaluate(ground_truth, text_search)
text_search_result

100%|██████████| 360/360 [00:00<00:00, 547.98it/s]


{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

#### Q5. Evaluating vector search

In [30]:
vector_search_result = evaluate(ground_truth, vector_search)
vector_search_result

100%|██████████| 360/360 [00:04<00:00, 84.99it/s] 


{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

##### Q6 — Tuning Hybrid Search

In [31]:
for k in [1, 50, 100, 200]:
  
    result = evaluate(ground_truth,
        lambda query,  k = k : hybrid_search(query,  k))
    print(f"boost={k}: {result}")
    #print(f"k={k}: MRR={result['mrr']:.4f}, Hit Rate={result['hit_rate']:.4f}")

100%|██████████| 360/360 [00:04<00:00, 74.04it/s]


boost=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


100%|██████████| 360/360 [00:05<00:00, 71.62it/s]


boost=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:04<00:00, 77.63it/s]


boost=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:05<00:00, 71.04it/s]

boost=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
